# Run the live TravelTime check

Run this notebook from the Databricks Git folder after preparing the real-data
lookup checkpoints through notebook 06b. This sends live TravelTime requests
for the three blood-bank searches and uses the account's API quota.

Run the first cell to create the settings. Set `mode`, `as_of` and `checkpoint_root`
to the same settings as 06b. If you left the upstream checkpoint root at its
default, leave this notebook's `checkpoint_root` blank. An empty `as_of` uses the
project configuration date. If the Git folder cannot be detected, set `repo_root`
to its absolute `/Workspace/...` path.

Use credentials already in the environment or set `secret_scope` to a Databricks
scope with keys named `TRAVELTIME_APP_ID` and `TRAVELTIME_API_KEY`. Run the second
cell to call the test and print the results.

The default receipt is `traveltime-live.json` in the chosen snapshot directory.
To save it elsewhere, set `receipt_path` to a writable absolute file path, such
as a file in a Unity Catalog volume. The full JSON also appears in the cell output.

In [ ]:
import os

defaults = {
    "repo_root": os.environ.get("DATATHON_REPO_ROOT", ""),
    "checkpoint_root": os.environ.get("DATATHON_OUTPUT_DIR", ""),
    "mode": os.environ.get("DATATHON_MODE", "databricks"),
    "as_of": "",
    "receipt_path": "",
    "secret_scope": "",
}
for name, default in defaults.items():
    dbutils.widgets.text(name, default)

In [ ]:
import json
from pathlib import Path
from runpy import run_path
import sys
from unittest.mock import patch

repo_setting = dbutils.widgets.get("repo_root").strip()
cwd = Path.cwd().resolve()
locations = (Path(repo_setting).expanduser(),) if repo_setting else (
    cwd, *cwd.parents, *(Path(path) for path in sys.path if path)
)
config_path = Path("non-ml/config/project.json")
root = next((folder.resolve() for folder in locations if (folder / config_path).is_file()), None)
if root is None:
    raise ValueError("Set repo_root to the absolute path of the Databricks Git folder for datathon-F26.")

config = json.loads((root / config_path).read_text(encoding="utf-8"))
checkpoint_root = root / (dbutils.widgets.get("checkpoint_root").strip() or "non-ml/outputs")
mode = dbutils.widgets.get("mode").strip()
as_of = dbutils.widgets.get("as_of").strip() or config["as_of"]
receipt_setting = dbutils.widgets.get("receipt_path").strip()
receipt = root / receipt_setting if receipt_setting else (
    checkpoint_root / mode / as_of / "traveltime-live.json"
)

def runner_environment():
    scope = dbutils.widgets.get("secret_scope").strip()
    environment = {"DATATHON_REPO_ROOT": str(root)}
    for key in ("TRAVELTIME_APP_ID", "TRAVELTIME_API_KEY"):
        value = os.environ.get(key, "").strip()
        if not value and scope:
            value = dbutils.secrets.get(scope=scope, key=key).strip()
        if not value:
            raise ValueError(f"Set {key} in the environment or in the Databricks secret scope.")
        environment[key] = value
    return environment

print(f"Running the live TravelTime check for {mode}/{as_of}. Receipt path: {receipt}", flush=True)
try:
    with patch.dict(os.environ, runner_environment()):
        run_check = run_path(str(root / "non-ml/check_traveltime_live.py"))["main"]
        status = run_check([
            "--checkpoint-root", str(checkpoint_root), "--mode", mode,
            "--as-of", as_of, "--output", str(receipt),
        ])
except SystemExit as error:
    raise RuntimeError("TravelTime could not start. See the error printed above.") from error

print(receipt.read_text(encoding="utf-8"))
if status:
    raise RuntimeError("At least one case failed or was inconclusive. See the saved receipt above.")